# Stage 4c — Where the model cannot decide

**One question.** At what confidence does the classifier stop being reliable, so commits below that
line can be set aside rather than forced into one of three classes?

**Why this is needed.** Levin's labelling had a fourth outcome that never reached the dataset: when
the authors could not classify a commit confidently they *dropped it and resampled*. So the training
data contains **no examples of unclassifiable commits**, and the model has never seen one. Faced
with a genuinely ambiguous commit it will still emit one of three labels, and often emit it
confidently.

That is a distribution problem, not only a confidence problem, and it is the reason a threshold has
to be found empirically rather than assumed.

**Calibrated on** the 173 Levin test commits — the only data this model has neither trained on nor
early-stopped against.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require, load_weights
WORK = setup()

## 1. Load the model and score the test commits

`model_attention_2024.pt`, the model that goes forward to Stage 5.

In [1]:
import json, re, math, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_score

# WORK comes from the setup cell; require() copies or downloads any input that is missing,
# including the trained weights from Hugging Face.
def find_input(name):
    return require(name, optional=True)

# model_attention_2024.pt if 04b wrote it to WORK, else the published .safetensors
MODEL_PATH = Path("model_attention_2024")

MODEL   = "microsoft/codebert-base"
MAXLEN  = 512
LABELS  = ["adaptive", "corrective", "perfective"]
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE  = None            # attention pooling used no overlap
print(f"model : {MODEL_PATH}")
print(f"device: {DEVICE}")

Mounted at /content/drive
model : /content/drive/MyDrive/levin_stage4b/model_attention_2024.pt
device: cuda


In [2]:
# --- rebuild the inputs exactly as Stage 4b did ---
D = pd.read_pickle(find_input("levin_dataset.pkl"))
R = pd.DataFrame([json.loads(l) for l in open(find_input("levin_with_diffs.jsonl"),
                                              encoding="utf-8") if l.strip()])
R = R[R["ok"] == True].reset_index(drop=True)
R["java_files"] = R["files"].map(
    lambda fs: sum(1 for f in (fs or []) if str(f.get("filename", "")).endswith(".java")))
R = R[(R["truncated"] != True) & (R["java_files"] > 0)].reset_index(drop=True)

ID_COL = json.load(open(find_input("levin_columns.json")))["hash"]
d2 = D.drop_duplicates(subset=[ID_COL], keep="first").reset_index(drop=True)
te_ids = set(d2.iloc[np.load(find_input("stage2_test_idx.npy"))][ID_COL])
R["is_test"] = R["commit_id"].isin(te_ids)

L2I = {l: i for i, l in enumerate(LABELS)}
Y = R["label"].map(L2I).to_numpy()
TEST_IDX = R.index[R.is_test].to_numpy()
print(f"{len(TEST_IDX)} test commits")

HUNK = re.compile(r"^@@ .*? @@")
def message_of(rec):
    return (rec.get("git_message") or rec.get("dataset_comment") or "").strip()

def structured_pieces(rec):
    out = [f"<COMMIT_MESSAGE>{message_of(rec)}</COMMIT_MESSAGE>"]
    files = []
    for f in rec.get("files") or []:
        added, removed = [], []
        for ln in (f.get("patch") or "").split("\n"):
            if HUNK.match(ln) or ln.startswith(("--- ", "+++ ")):
                continue
            if ln.startswith("+"):
                added.append(ln[1:])
            elif ln.startswith("-"):
                removed.append(ln[1:])
        piece = [f"<FILE>{f.get('filename')}</FILE>"]
        if removed:
            piece.append("<REMOVED>" + "\n".join(removed) + "</REMOVED>")
        if added:
            piece.append("<ADDED>" + "\n".join(added) + "</ADDED>")
        files.append("\n".join(piece))
    files.sort(key=len, reverse=True)
    return out + files

PIECES = R.apply(structured_pieces, axis=1)
tok = AutoTokenizer.from_pretrained(MODEL)


class CommitClassifier(nn.Module):
    """Identical to Stage 4b -- the state dict must load into the same definition."""

    def __init__(self, pooling="attention", dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL)
        hid = self.enc.config.hidden_size
        self.pooling = pooling
        if pooling == "attention":
            self.score = nn.Linear(hid, 1)
        else:
            layer = nn.TransformerEncoderLayer(hid, nhead=4, dim_feedforward=512,
                                               dropout=dropout, batch_first=True)
            self.tr = nn.TransformerEncoder(layer, num_layers=1)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid, len(LABELS))

    def forward(self, ids, att, commit_of, piece_of, n_commits):
        h = self.enc(input_ids=ids, attention_mask=att).last_hidden_state[:, 0]
        key = commit_of * 100_000 + piece_of
        uniq, inv = torch.unique(key, return_inverse=True)
        pv = torch.zeros(len(uniq), h.size(1), device=h.device, dtype=h.dtype)
        pv.index_add_(0, inv, h)
        cnt = torch.zeros(len(uniq), device=h.device, dtype=h.dtype)
        cnt.index_add_(0, inv, torch.ones(len(h), device=h.device, dtype=h.dtype))
        pv = pv / cnt.unsqueeze(1)
        pc = (uniq // 100_000).long()
        per = torch.bincount(pc, minlength=n_commits)
        P = int(per.max().item())
        x = torch.zeros(n_commits, P, h.size(1), device=h.device, dtype=h.dtype)
        m = torch.zeros(n_commits, P, device=h.device, dtype=h.dtype)
        slot = torch.zeros(n_commits, dtype=torch.long, device=h.device)
        for i in range(len(pc)):
            c = pc[i]; x[c, slot[c]] = pv[i]; m[c, slot[c]] = 1; slot[c] += 1
        mm = m.unsqueeze(-1)
        if self.pooling == "attention":
            a = self.score(x).masked_fill(mm == 0, -1e9).softmax(1)
            v = (x * a).sum(1)
        else:
            hh = self.tr(x, src_key_padding_mask=(m == 0))
            v = (hh * mm).sum(1) / mm.sum(1).clamp(min=1)
        return self.out(self.drop(v))


model = CommitClassifier("attention").to(DEVICE)
model.load_state_dict(load_weights("model_attention_2024", DEVICE))
model.eval()
print("model loaded")

173 test commits


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

model loaded


In [3]:
def windows(text, stride=STRIDE):
    ids = tok.encode(text, add_special_tokens=False, truncation=False)
    body = MAXLEN - 2
    if len(ids) <= body:
        return [ids]
    step = body if stride is None else stride
    return [ids[i:i + body] for i in range(0, len(ids), step) if i < len(ids)]


@torch.no_grad()
def logits_for(idx, max_windows=96):
    """Raw logits per commit, one commit at a time so piece count cannot blow memory."""
    out = []
    for n, ci in enumerate(idx):
        wins, owner = [], []
        for pi, p in enumerate(PIECES.iloc[ci]):
            for w in windows(p):
                wins.append(w); owner.append(pi)
        # a very large commit is encoded in chunks of windows, pieces preserved
        L = min(MAXLEN, max(len(w) for w in wins) + 2)
        ids = torch.full((len(wins), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(wins), L), dtype=torch.long)
        for i, w in enumerate(wins):
            seq = [tok.cls_token_id] + w[:L - 2] + [tok.sep_token_id]
            ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
        lg = model(ids.to(DEVICE), att.to(DEVICE),
                   torch.zeros(len(wins), dtype=torch.long, device=DEVICE),
                   torch.tensor(owner, device=DEVICE), 1)
        out.append(lg.float().cpu().numpy()[0])
        if n % 25 == 0:
            print(f"\r   {n}/{len(idx)}", end="", flush=True)
    print(f"\r   {len(idx)}/{len(idx)} scored" + " " * 10)
    return np.stack(out)


LOGITS = logits_for(TEST_IDX)
TRUE = Y[TEST_IDX]
pred = LOGITS.argmax(1)
print(f"\naccuracy {accuracy_score(TRUE, pred)*100:.1f}%   "
      f"kappa {cohen_kappa_score(TRUE, pred)*100:.1f}")
print("matches Stage 4b, so the model reloaded correctly")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4884 > 512). Running this sequence through the model will result in indexing errors


   173/173 scored          

accuracy 87.9%   kappa 81.0
matches Stage 4b, so the model reloaded correctly


## 2. Two ways to measure confidence

The model outputs three probabilities summing to one.

**Max probability** is the highest of the three. **Margin** is the top minus the second.

They differ in a way that matters here:

| | corrective | adaptive | perfective | max | margin |
|---|---|---|---|---|---|
| A | 0.70 | 0.20 | 0.10 | 0.70 | **0.50** |
| B | 0.70 | 0.28 | 0.02 | 0.70 | **0.42** |
| C | 0.38 | 0.35 | 0.27 | 0.38 | **0.03** |

A and B have identical max probability, but B is a near-tie between two classes while A is clear.
Margin separates *confidently corrective* from *torn between corrective and adaptive* — which is
closer to what "the model cannot decide" means.

In [4]:
def softmax(z, T=1.0):
    z = z / T
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def confidence(probs):
    s = np.sort(probs, axis=1)
    return s[:, -1], s[:, -1] - s[:, -2]        # max, margin


P_raw = softmax(LOGITS)
maxp_raw, marg_raw = confidence(P_raw)
correct = (pred == TRUE)

print("RAW SOFTMAX\n")
print(pd.DataFrame({"max prob": maxp_raw, "margin": marg_raw})
      .describe(percentiles=[.1, .25, .5, .75]).round(3).to_string())
print(f"\nmean confidence {maxp_raw.mean():.3f}   actual accuracy {correct.mean():.3f}")
gap = maxp_raw.mean() - correct.mean()
print(f"gap {gap:+.3f}")
print("""
A neural network's softmax is not a probability. If the mean confidence sits well above the actual
accuracy, the model is overconfident and a threshold placed on these numbers would mean something
other than what it says.""")

RAW SOFTMAX

       max prob   margin
count   173.000  173.000
mean      0.976    0.954
std       0.071    0.139
min       0.520    0.042
10%       0.933    0.869
25%       0.995    0.990
50%       1.000    0.999
75%       1.000    1.000
max       1.000    1.000

mean confidence 0.976   actual accuracy 0.879
gap +0.098

A neural network's softmax is not a probability. If the mean confidence sits well above the actual
accuracy, the model is overconfident and a threshold placed on these numbers would mean something
other than what it says.


## 3. Calibration

**Temperature scaling** (Guo et al., ICML 2017): divide the logits by a single learned scalar before
the softmax. One parameter, fitted by minimising negative log-likelihood.

It cannot change any prediction — dividing all three logits by the same number leaves the ordering
untouched — so accuracy and kappa are unaffected. Only the confidence values move, which is exactly
what a threshold needs.

**Expected Calibration Error** measures the result: bin the predictions by confidence and compare
each bin's average confidence against its actual accuracy. A well-calibrated model sits on the
diagonal; ECE is the weighted average distance from it.

In [5]:
def ece(probs, true, bins=8):
    """Expected calibration error, plus the per-bin table behind it."""
    conf = probs.max(axis=1)
    pred_ = probs.argmax(1)
    hit = (pred_ == true).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    rows, total = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if not m.any():
            continue
        rows.append({"bin": f"{lo:.2f}-{hi:.2f}", "n": int(m.sum()),
                     "confidence": round(conf[m].mean(), 3),
                     "accuracy": round(hit[m].mean(), 3),
                     "gap": round(conf[m].mean() - hit[m].mean(), 3)})
        total += m.mean() * abs(conf[m].mean() - hit[m].mean())
    return total, pd.DataFrame(rows)


def fit_temperature(logits, true, grid=np.arange(0.5, 5.01, 0.01)):
    """The T minimising negative log-likelihood. One parameter, so little room to overfit."""
    best_T, best_nll = 1.0, np.inf
    for T in grid:
        p = softmax(logits, T)
        nll = -np.log(np.clip(p[np.arange(len(true)), true], 1e-12, None)).mean()
        if nll < best_nll:
            best_T, best_nll = float(T), nll
    return best_T, best_nll


e_raw, tab_raw = ece(P_raw, TRUE)
T, nll = fit_temperature(LOGITS, TRUE)
P_cal = softmax(LOGITS, T)
e_cal, tab_cal = ece(P_cal, TRUE)

print(f"temperature T = {T:.2f}")
print(f"   T > 1 means the model was overconfident and is being softened")
print(f"   T < 1 means it was underconfident\n")
print(f"ECE before {e_raw:.3f}   after {e_cal:.3f}   ({e_cal - e_raw:+.3f})")
print(f"accuracy unchanged: {accuracy_score(TRUE, P_cal.argmax(1))*100:.1f}%")

print("\n\nRELIABILITY, before calibration\n")
print(tab_raw.to_string(index=False))
print("\nRELIABILITY, after calibration\n")
print(tab_cal.to_string(index=False))
print("""
'gap' is confidence minus accuracy within each bin. Positive means overconfident. After
calibration the gaps should be smaller and less one-sided.""")

maxp, marg = confidence(P_cal)

temperature T = 2.86
   T > 1 means the model was overconfident and is being softened
   T < 1 means it was underconfident

ECE before 0.109   after 0.025   (-0.083)
accuracy unchanged: 87.9%


RELIABILITY, before calibration

      bin   n  confidence  accuracy    gap
0.50-0.62   3       0.579     0.000  0.579
0.62-0.75   2       0.725     0.500  0.225
0.75-0.88   5       0.808     1.000 -0.192
0.88-1.00 163       0.992     0.896  0.096

RELIABILITY, after calibration

      bin   n  confidence  accuracy    gap
0.38-0.50   3       0.481     0.667 -0.186
0.50-0.62   8       0.561     0.500  0.061
0.62-0.75  18       0.686     0.778 -0.092
0.75-0.88  29       0.817     0.862 -0.045
0.88-1.00 115       0.934     0.930  0.003

'gap' is confidence minus accuracy within each bin. Positive means overconfident. After
calibration the gaps should be smaller and less one-sided.


## 4. Where to draw the line

For each candidate threshold: of the commits the model keeps, how many does it get right
(**precision**), and how many does it keep at all (**coverage**).

The trade is direct — a stricter threshold means cleaner labels on fewer commits.

In [6]:
TARGET = 0.90        # required precision among kept commits

def sweep(score, name):
    rows = []
    for t in np.arange(0.0, 1.0, 0.02):
        keep = score >= t
        if keep.sum() < 10:
            break
        rows.append({"threshold": round(float(t), 2),
                     "kept": int(keep.sum()),
                     "coverage %": round(keep.mean() * 100, 1),
                     "precision %": round(correct[keep].mean() * 100, 1),
                     "set aside": int((~keep).sum())})
    df = pd.DataFrame(rows)
    ok = df[df["precision %"] >= TARGET * 100]
    chosen = ok.iloc[0] if len(ok) else None
    print(f"\n{name.upper()}\n")
    print(df.iloc[::3].to_string(index=False))
    if chosen is not None:
        print(f"\n   lowest threshold reaching {TARGET*100:.0f}% precision: "
              f"{chosen['threshold']:.2f}")
        print(f"   keeps {chosen['kept']} of {len(score)} "
              f"({chosen['coverage %']:.1f}%), sets aside {chosen['set aside']}")
    else:
        print(f"\n   no threshold reaches {TARGET*100:.0f}% precision on this sample")
    return df, chosen


sweep_max, best_max = sweep(maxp, "calibrated max probability")
sweep_marg, best_marg = sweep(marg, "margin")


CALIBRATED MAX PROBABILITY

 threshold  kept  coverage %  precision %  set aside
      0.00   173       100.0         87.9          0
      0.06   173       100.0         87.9          0
      0.12   173       100.0         87.9          0
      0.18   173       100.0         87.9          0
      0.24   173       100.0         87.9          0
      0.30   173       100.0         87.9          0
      0.36   173       100.0         87.9          0
      0.42   173       100.0         87.9          0
      0.48   172        99.4         88.4          1
      0.54   167        96.5         89.2          6
      0.60   165        95.4         89.7          8
      0.66   156        90.2         91.0         17
      0.72   148        85.5         91.2         25
      0.78   138        79.8         92.0         35
      0.84   127        73.4         92.1         46
      0.90   102        59.0         93.1         71
      0.96    12         6.9         83.3        161

   lowest thresh

In [7]:
# Which of the two separates right from wrong more sharply.
print("SEPARATION BETWEEN CORRECT AND INCORRECT PREDICTIONS\n")
print(f"{'':<22}{'correct':>10}{'incorrect':>12}{'difference':>13}")
print("-" * 57)
for name, s in (("calibrated max prob", maxp), ("margin", marg)):
    a, b = s[correct].mean(), s[~correct].mean()
    print(f"{name:<22}{a:>10.3f}{b:>12.3f}{a-b:>13.3f}")

from sklearn.metrics import roc_auc_score
print("\nability to predict its own errors (AUC, 0.5 = no signal)\n")
for name, s in (("calibrated max prob", maxp), ("margin", marg)):
    print(f"   {name:<22}{roc_auc_score(correct, s):.3f}")
print("""
Higher is better. Whichever separates more cleanly is the one to threshold on; if they are close,
requiring both is stricter than either alone and costs nothing extra.""")

SEPARATION BETWEEN CORRECT AND INCORRECT PREDICTIONS

                         correct   incorrect   difference
---------------------------------------------------------
calibrated max prob        0.877       0.765        0.112
margin                     0.791       0.586        0.205

ability to predict its own errors (AUC, 0.5 = no signal)

   calibrated max prob   0.703
   margin                0.704

Higher is better. Whichever separates more cleanly is the one to threshold on; if they are close,
requiring both is stricter than either alone and costs nothing extra.


## 5. The boundary, and what it means

The threshold carried into Stage 5. Commits scoring below it are assigned to a fourth bucket rather
than forced into one of three classes.

In [8]:
use_margin = roc_auc_score(correct, marg) >= roc_auc_score(correct, maxp)
score, best, which = ((marg, best_marg, "margin") if use_margin
                      else (maxp, best_max, "calibrated max probability"))

if best is None:
    raise SystemExit(f"No threshold reaches {TARGET*100:.0f}% precision. Either lower TARGET or "
                     f"accept that this model cannot be made that reliable on this sample.")

THRESH = float(best["threshold"])
keep = score >= THRESH

print("=" * 66)
print("BOUNDARY FOR STAGE 5")
print("=" * 66)
print(f"   temperature      T = {T:.2f}")
print(f"   score            {which}")
print(f"   threshold        {THRESH:.2f}")
print(f"   precision above  {best['precision %']:.1f}%")
print(f"   coverage         {best['coverage %']:.1f}%")
print()
print("WHAT THE FOURTH BUCKET WOULD CONTAIN, on this test set\n")
below = ~keep
print(f"   {int(below.sum())} commits ({below.mean()*100:.1f}%)")
if below.any():
    print(f"   of those the model would have got right : "
          f"{int(correct[below].sum())} ({correct[below].mean()*100:.1f}%)")
    print(f"   true class among them: " +
          ", ".join(f"{LABELS[i]} {int((TRUE[below]==i).sum())}" for i in range(3)))

json.dump({"model": str(MODEL_PATH.name), "temperature": round(T, 4),
           "score": which, "threshold": THRESH,
           "target_precision": TARGET,
           "precision_at_threshold": float(best["precision %"]),
           "coverage_at_threshold": float(best["coverage %"]),
           "ece_before": round(float(e_raw), 4), "ece_after": round(float(e_cal), 4),
           "calibration_n": int(len(TEST_IDX))},
          open(WORK / "stage4c_calibration.json", "w"), indent=1)
print(f"\n   {WORK/'stage4c_calibration.json'} written")

BOUNDARY FOR STAGE 5
   temperature      T = 2.86
   score            margin
   threshold        0.34
   precision above  90.7%
   coverage         93.1%

WHAT THE FOURTH BUCKET WOULD CONTAIN, on this test set

   12 commits (6.9%)
   of those the model would have got right : 6 (50.0%)
   true class among them: adaptive 4, corrective 8, perfective 0

   /content/drive/MyDrive/levin_stage4b/stage4c_calibration.json written


### Three limits to carry forward

**The calibration set is 173 commits**, and this model was selected on them — seed 2024 scored 87.9%
against a three-seed mean of 86.7%. The threshold is therefore optimistic, probably by a little more
than the bin noise suggests.

**Levin's data was pre-filtered for legibility.** Commits the authors could not confidently classify
were dropped and resampled. A threshold calibrated on that population will be too permissive on a
corpus that includes terse, uninformative commits — which is most corpora, including this study's.

**So the fourth bucket needs checking by hand.** Take fifty commits that fall below the line in Stage
5 and read them. If they are genuinely ambiguous — terse, mixed intent, no clue either way — the
bucket means what it claims. If they are clearly classifiable and the model simply struggled, the
honest label is "low confidence", not "unclassifiable", and the write-up should say so.

Levin's own inter-rater agreement was **79% before shared guidelines**, so roughly one commit in five
is contestable between two humans. A fourth bucket in that range is plausible; one at 60% means
something else is wrong.